In [1]:
import nflreadpy as nfl
import polars as pl
import gc

In [2]:
schedules = nfl.load_schedules(seasons = list(range(2012, 2026)))

In [3]:
pbp = nfl.load_pbp(seasons = list(range(2012, 2026)))

In [4]:
schedules_columns = [
    "game_id",
    "season",
    "week",
    "game_type",
    "gameday",
    "gametime",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "location"
]

available_columns = [
    col for col in schedules_columns
    if col in schedules.columns
]

schedules_clean = (
    schedules
    .select(available_columns)
    .filter(
        pl.col("home_score").is_not_null() & pl.col("away_score").is_not_null()
    )
    .with_columns(
        pl.when(pl.col("home_score") > pl.col("away_score"))
        .then(1)
        .when(pl.col("home_score") < pl.col("away_score"))
        .then(0)
        .otherwise(None)
        .alias("home_win")
    )

    .filter(
        (pl.col("game_type") == "REG") & pl.col("home_win").is_not_null()
    )
    .sort(["season", "week", "gameday", "gametime"])
)

del schedules
gc.collect()

33

In [5]:
pbp_columns = [
    "game_id",
    "play_id",
    "season",
    "week",
    "season_type",
    "game_date",
    "posteam",
    "defteam",
    "home_team",
    "away_team",
    "play_type",
    "epa",
    "success",
    "yards_gained",
    "pass_attempt",
    "rush_attempt",
    "qb_dropback",
    "qb_scramble",
    "sack",
    "interception",
    "fumble_lost",
    "touchdown",
    "pass_touchdown",
    "rush_touchdown",
    "first_down",
    "down",
    "ydstogo",
    "yardline_100",
    "shotgun",
    "no_huddle",
    "complete_pass",
    "air_yards",
    "yards_after_catch",
    "penalty",
    "penalty_yards",
    "third_down_converted",
    "fourth_down_converted",
    "field_goal_attempt",
    "punt_attempt",
    "drive",
    "drive_result",
]

available_pbp_columns = [
    col for col in pbp_columns
    if col in pbp.columns
]

pbp_clean = (
    pbp
    .select(available_pbp_columns)
    .filter(
        (pl.col("season_type") == "REG") & pl.col("play_type").is_not_null()
    )
)

del pbp
gc.collect()

33

In [6]:
offensive_plays = pbp_clean.filter(
    pl.col("posteam").is_not_null()
    & pl.col("defteam").is_not_null()
)

pbp_team_game = (
    offensive_plays
    .group_by(["game_id", "season", "week", "posteam", "defteam"])
    .agg(
        pl.len().alias("offensive_plays"),

        pl.col("epa").mean().alias("epa_per_play"),
        pl.col("success").mean().alias("success_rate"),
        pl.col("yards_gained").mean().alias("yards_per_play"),
        pl.col("pass_attempt").fill_null(0).sum().alias("pass_attempts"),
        pl.col("rush_attempt").fill_null(0).sum().alias("rush_attempts"),
        pl.col("qb_dropback").fill_null(0).sum().alias("dropbacks"),
        pl.col("sack").fill_null(0).sum().alias("sacks"),
        pl.col("interception").fill_null(0).sum().alias("interceptions"),
        pl.col("fumble_lost").fill_null(0).sum().alias("fumbles_lost"),
        pl.col("touchdown").fill_null(0).sum().alias("touchdowns"),
        pl.col("pass_touchdown").fill_null(0).sum().alias("pass_touchdowns"),
        pl.col("rush_touchdown").fill_null(0).sum().alias("rush_touchdowns"),
        pl.col("first_down").fill_null(0).sum().alias("first_downs"),
        pl.col("shotgun").fill_null(0).sum().alias("shotgun_plays"),
        pl.col("no_huddle").fill_null(0).sum().alias("no_huddle_plays"),
        pl.col("third_down_converted").fill_null(0).sum()
        .alias("third_down_conversions"),
        pl.col("fourth_down_converted").fill_null(0).sum()
        .alias("fourth_down_conversions"),
    )
)

In [7]:
offense = pbp_team_game.select(
    "game_id",
    "season",
    "week",
    pl.col("posteam").alias("team"),
    pl.col("defteam").alias("opponent"),
    "offensive_plays",
    "epa_per_play",
    "success_rate",
    "yards_per_play",
    "pass_attempts",
    "rush_attempts",
    "dropbacks",
    "sacks",
    "interceptions",
    "fumbles_lost",
    "touchdowns",
    "pass_touchdowns",
    "rush_touchdowns",
    "first_downs",
    "shotgun_plays",
    "no_huddle_plays",
    "third_down_conversions",
    "fourth_down_conversions",
)

In [8]:
defense = pbp_team_game.select(
    "game_id",
    pl.col("defteam").alias("team"),

    pl.col("offensive_plays").alias("plays_faced"),
    pl.col("epa_per_play").alias("epa_allowed_per_play"),
    pl.col("success_rate").alias("success_rate_allowed"),
    pl.col("yards_per_play").alias("yards_allowed_per_play"),

    (
        pl.col("interceptions") + pl.col("fumbles_lost")
    ).alias("turnovers_forced"),

    pl.col("touchdowns").alias("touchdowns_allowed"),
)

In [9]:
team_game_stats = (
    offense
    .join(defense, on=["game_id", "team"], how="left")
    .sort(["season", "week", "game_id", "team"])
)

In [10]:
del offense
del defense
del pbp_clean
del pbp_team_game
del offensive_plays
gc.collect()

34